# Task 1: Parse Text Data into Structured Format

## Step 1: Initialize Spark Context

In [8]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()
print(f" Spark {sc.version} ready")

 Spark 3.5.0 ready


## Step 2: Read Raw File

In [9]:
raw_rdd = sc.textFile("/home/jovyan/data/raw/employees.txt")

print("Raw lines count:", raw_rdd.count())
print("\nFirst 5 lines:")
for line in raw_rdd.take(5):
    print(repr(line))

Raw lines count: 13

First 5 lines:
'emp_id,name,department,job_title,salary,location,hire_date,performance_rating,years_exp'
'1,John Smith,Engineering,Senior Developer,125000,San Francisco,2021-03-15,4.5,8'
'2,Sarah Johnson,Sales,Account Executive,85000,New York,2022-01-10,4.2,5'
'3,Michael Williams,Engineering,Software Engineer,95000,Austin,2023-06-20,3.8,3'
'4,Jennifer Brown,Marketing,Marketing Manager,92000,Chicago,2020-11-05,4.7,7'


## Step 3: Remove Header & Empty Lines

In [10]:
header = raw_rdd.first()

data_rdd = raw_rdd.filter(lambda line: line != header) \
                  .filter(lambda line: line.strip() != "")

print("Data lines (no header, no empties):", data_rdd.count())

Data lines (no header, no empties): 10


## Step 4: Parse CSV Lines into Lists

In [11]:
parsed_rdd = data_rdd.map(lambda line: line.split(","))

print("Sample parsed record:")
print(parsed_rdd.first())

Sample parsed record:
['1', 'John Smith', 'Engineering', 'Senior Developer', '125000', 'San Francisco', '2021-03-15', '4.5', '8']


## Step 5: Separate Valid & Invalid Records

Expected schema: 9 fields per record.

In [12]:
EXPECTED_COLS = 9

def validate(record):
    """Valid if exactly 9 fields AND salary field is numeric."""
    if len(record) != EXPECTED_COLS:
        return False
    # Additional check: salary (index 4) must be numeric
    try:
        float(record[4])
        return True
    except ValueError:
        return False

valid_rdd = parsed_rdd.filter(validate)
invalid_rdd = parsed_rdd.filter(lambda r: not validate(r))

print(f" Valid:   {valid_rdd.count()}")
print(f" Invalid: {invalid_rdd.count()}")

 Valid:   9
 Invalid: 1


## Step 6: Inspect Invalid Records

In [13]:
print("Invalid records:")
for rec in invalid_rdd.collect():
    print(f"  Fields: {len(rec)} | {rec}")

Invalid records:
  Fields: 9 | ['7', 'Robert Martinez', 'Legal', 'Legal Counsel145000', 'San Francisco', '2019-09-22', '4.8', '10', ' 5']


## Step 7: Display Valid Structured Data

In [ ]:
print("Valid employee records:\n")
for rec in valid_rdd.collect():
    print(f"  ID: {rec[0]:<3} Name: {rec[1]:<20} Dept: {rec[2]:<15} Salary: ${rec[4]}")

Valid employee records:

  ID: 1   Name: John Smith           Dept: Engineering     Salary: $125000
  ID: 2   Name: Sarah Johnson        Dept: Sales           Salary: $85000
  ID: 3   Name: Michael Williams     Dept: Engineering     Salary: $95000
  ID: 4   Name: Jennifer Brown       Dept: Marketing       Salary: $92000
  ID: 5   Name: David Jones          Dept: Finance         Salary: $105000
  ID: 6   Name: Lisa Garcia          Dept: IT              Salary: $115000
  ID: 8   Name: Patricia Wilson      Dept: HR              Salary: $88000
  ID: 9   Name: James Anderson       Dept: Sales           Salary: $110000
  ID: 10  Name: Mary Thomas          Dept: Engineering     Salary: $145000


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 47118)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

##  Result

Data parsed successfully. Valid records ready for further tasks.

##  Task 1 Complete

| Metric | Count |
|--------|-------|
| Raw lines | 13 |
| Valid records | 9 |
| Invalid records | 1 |

**Invalid Record:** Line 7 (Robert Martinez) - missing comma between `job_title` and `salary`.

Valid records ready for downstream tasks.